# exp025_pseudo_tail_postprocess_cv_audit train

Regenerate fold-safe OOF predictions for the exp024 raw pseudo-tail anchor and audit conservative postprocess candidates with held-out folds.


## Contents

1. Setup and configuration
2. Pseudo-tail OOF regeneration
3. Postprocess CV audit
4. Metrics and artifacts


## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from pseudo_tail_postprocess_cv_audit import get_nested, load_yaml, run_audit, train_files
from settings import ExperimentPaths

DEBUG_MAX_WELLS = None

paths = ExperimentPaths()
config = load_yaml(Path("config.yaml"))
output_dir = paths.artifacts_dir
output_dir.mkdir(parents=True, exist_ok=True)
files = train_files(paths, DEBUG_MAX_WELLS)

print(f"Experiment: {config['experiment']['name']}")
print(f"Train wells: {len(files)}")
print(f"Parent best variant: {get_nested(config, 'audit.parent_best_variant')}")
print(f"Parent clean CV: {get_nested(config, 'audit.parent_best_cv')}")
print(f"Parent Public LB: {get_nested(config, 'audit.parent_public_lb')}")
print(f"Fixed candidates: {len(get_nested(config, 'audit.fixed_candidates', []))}")


## 2. Pseudo-tail OOF regeneration

Fit the selected `pseudo_tail_3_cutoffs_distance_balanced` recipe on each training fold. Pseudo cutoffs are created only inside train-fold wells; held-out wells use their original `TVT_input` tail.

## 3. Postprocess CV audit

The audit compares raw pseudo-tail predictions, fixed shrink candidates, same-OOF bucket alpha fitting, original-fold-held alpha fitting, and well-hash-held alpha fitting.

In [ ]:
summary = run_audit(files=files, config=config, output_dir=output_dir)
print(json.dumps(summary, indent=2, sort_keys=True))


## 4. Metrics and artifacts

In [ ]:
metrics = pd.read_csv(output_dir / "pseudo_tail_postprocess_metrics.csv")
folds = pd.read_csv(output_dir / "pseudo_tail_postprocess_fold_metrics.csv")
buckets = pd.read_csv(output_dir / "pseudo_tail_postprocess_bucket_summary.csv")

display(metrics.sort_values("rmse"))
display(folds)
display(buckets[buckets["candidate"].isin(["raw_pseudo_tail", "same_oof_bucket_alpha_fit"])])

print("Selected method:", summary["selected_method"])
print("Selected clean CV:", summary["selected_clean_cv"])
print("Clean postprocess supported:", summary["clean_postprocess_supported"])
